# 15 - Word2Vec with Negative Sampling

Goal: Implement Word2Vec efficiently using negative sampling

Run with: conda activate tfenv

In [21]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import plotly.express as px
import pandas as pd
import time

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.21.0


In [22]:
# Load dataset using HuggingFace datasets API (as in notebook 14)
print("Loading gaianet/london dataset...")
from datasets import load_dataset

ds = load_dataset("gaianet/london", split="train")
print(f"Dataset size: {len(ds)}")

Loading gaianet/london dataset...
Dataset size: 661


In [23]:
texts = [row['text'] if 'text' in row else row.get('content', '') for row in ds][:50000]
full_text = ' '.join(texts)
print(f"Total chars: {len(full_text)}")

# Build vocabulary and preprocess words as in notebook 14
words = full_text.lower().split()
words = [w.strip('.,;:!?()[]\"\'-0123456789') for w in words]
words = [w for w in words if len(w) > 1]
print(f"Total words: {len(words)}")
print(f"Sample: {words[:20]}")

Total chars: 186582
Total words: 28132
Sample: ['london', 'is', 'the', 'capital', 'and', 'largest', 'city', 'of', 'england', 'and', 'the', 'united', 'kingdom', 'with', 'population', 'of', 'around', 'million', 'and', 'the']


In [24]:
# Build vocabulary (threshold 5 as in notebook 14)
from collections import Counter
word_counts = Counter(words)
vocab = [w for w, c in word_counts.items() if c >= 2]
text_vocab = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in text_vocab.items()}
text_vocab_size = len(text_vocab)
text_embed_dim = 64
print(f"Vocabulary: {text_vocab_size} words")

Vocabulary: 3291 words


In [25]:
# Create training pairs (Skip-gram)
def create_pairs(words, window=2):
    pairs = []
    for i, word in enumerate(words):
        if word not in text_vocab:
            continue
        for j in range(max(0, i - window), min(len(words), i + window + 1)):
            if i != j and words[j] in text_vocab:
                pairs.append((text_vocab[word], text_vocab[words[j]]))
    return pairs

pairs = create_pairs(words)
train_words = np.array([p[0] for p in pairs], dtype=np.int32)
train_context = np.array([p[1] for p in pairs], dtype=np.int32)
print(f"Training pairs: {len(pairs)}")

Training pairs: 112368


In [26]:
# Generate negative samples
NEGATIVE_SAMPLING = 5

all_indices = np.arange(text_vocab_size)

np.random.seed(42)
negative_samples = []

for target_idx in train_context:
    neg_indices = np.random.choice(all_indices, size=NEGATIVE_SAMPLING, replace=False)
    for neg_idx in neg_indices:
        negative_samples.append((target_idx, neg_idx))

neg_words = np.array([p[0] for p in negative_samples], dtype=np.int32)
neg_context = np.array([p[1] for p in negative_samples], dtype=np.int32)
neg_labels = np.zeros(len(negative_samples), dtype=np.float32)

print(f"NEGATIVE_SAMPLING = {NEGATIVE_SAMPLING}")
print(f"Generated {len(negative_samples)} negative samples")

KeyboardInterrupt: 

In [ ]:
# Combine positive and negative samples
pos_labels = np.ones(len(train_words), dtype=np.float32)

all_words = np.concatenate([train_words, neg_words])
all_context = np.concatenate([train_context, neg_context])
all_labels = np.concatenate([pos_labels, neg_labels])

print(f"Total training samples: {len(all_words)}")
print(f"  Positive (1): {len(pos_labels)}")
print(f"  Negative (0): {len(neg_labels)}")

Total training samples: 674208
  Positive (1): 112368
  Negative (0): 561840


In [ ]:
# Word2Vec with Negative Sampling - Custom Model
class Word2VecNS(keras.Model):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.target_embedding = layers.Embedding(
            vocab_size, embed_dim, name='embedding_target'
        )
        self.context_embedding = layers.Embedding(
            vocab_size, embed_dim, name='embedding_context'
        )
        # No Dense layer here, output is scalar logit
    def call(self, inputs):
        target, context = inputs
        target_emb = self.target_embedding(target)  # (batch, embed_dim)
        context_emb = self.context_embedding(context)  # (batch, embed_dim)
        dot_product = tf.reduce_sum(target_emb * context_emb, axis=-1, keepdims=True)  # (batch, 1)
        return tf.sigmoid(dot_product)  # (batch, 1)

model = Word2VecNS(text_vocab_size, text_embed_dim)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
    )

print("Word2Vec with Negative Sampling model ready!")
model.summary()

Word2Vec with Negative Sampling model ready!


E0000 00:00:1779755176.758652   31640 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1779755176.781113   31640 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "word2_vec_ns"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_target (Embedding)    │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_context (Embedding)   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Training with negative sampling
train_ds = tf.data.Dataset.from_tensor_slices(((all_words, all_context), all_labels))
train_ds = train_ds.shuffle(buffer_size=len(all_words)).batch(256).prefetch(tf.data.AUTOTUNE)

print("Training Word2Vec with Negative Sampling...")
start = time.time()

# model.fit expects a tuple (inputs, labels) for custom models
history = model.fit(
    train_ds,
    epochs=20,
    verbose=1
)

print(f"Training time: {time.time() - start:.2f}s")

Training Word2Vec with Negative Sampling...
Epoch 1/20


2634/2634 ━━━━━━━━━━━━━━━━━━━━ 19s 6ms/step - accuracy: 0.8883 - loss: 0.3066
Epoch 2/20
2634/2634 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - accuracy: 0.9389 - loss: 0.1634
Epoch 3/20
2634/2634 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - accuracy: 0.9498 - loss: 0.1348
Epoch 4/20
2634/2634 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - accuracy: 0.9522 - loss: 0.1296
Epoch 5/20
2634/2634 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - accuracy: 0.9533 - loss: 0.1274
Epoch 6/20
2634/2634 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.9537 - loss: 0.1268
Epoch 7/20
2634/2634 ━━━━━━━━━━━━━━━━━━━━ 18s 6ms/step - accuracy: 0.9543 - loss: 0.1258
Epoch 8/20
2634/2634 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - accuracy: 0.9546 - loss: 0.1253
Epoch 9/20
2634/2634 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - accuracy: 0.9547 - loss: 0.1246
Epoch 10/20
2634/2634 ━━━━━━━━━━━━━━━━━━━━ 17s 6ms/step - accuracy: 0.9545 - loss: 0.1250
Epoch 11/20
2634/2634 ━━━━━━━━━━━━━━━━━━━━ 19s 6ms/step - accuracy: 0.9552 - loss: 0.1241
Epoch 12/20
2634/2634 ━━━━━━━━

In [ ]:
# Get embeddings (use target embedding)
target_embeddings = model.target_embedding.get_weights()[0]
context_embeddings = model.context_embedding.get_weights()[0]

final_embeddings = (target_embeddings + context_embeddings) / 2

print(f"Target embeddings shape: {target_embeddings.shape}")
print(f"Context embeddings shape: {context_embeddings.shape}")

Target embeddings shape: (3291, 64)
Context embeddings shape: (3291, 64)


In [27]:
# Find similar words
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b) + 1e-8)

def find_similar(word, top_n=5):
    if word not in text_vocab:
        return []
    word_idx = text_vocab[word]
    word_emb = final_embeddings[word_idx]
    
    similarities = []
    for w, idx in text_vocab.items():
        if w != word:
            sim = cosine_similarity(word_emb, final_embeddings[idx])
            similarities.append((w, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

print("=== Similar Words (Negative Sampling) ===")
test_words = ['london', 'city', 'river', 'bridge', 'king', 'queen']
for word in test_words:
    if word in text_vocab:
        similar = find_similar(word)
        print(f"'{word}' -> {[(w, f'{s:.3f}') for w, s in similar]}")

=== Similar Words (Negative Sampling) ===
'london' -> [('the', '0.749'), ('of', '0.732'), ('in', '0.710'), ('and', '0.676'), ('city', '0.567')]
'city' -> [('in', '0.612'), ('the', '0.583'), ('london', '0.567'), ('and', '0.473'), ('of', '0.438')]
'river' -> [('thames', '0.592'), ('flows', '0.579'), ('crosses', '0.566'), ('effra', '0.517'), ('navigable', '0.476')]
'bridge' -> [('hundred', '0.565'), ('would', '0.472'), ('trafalgar', '0.444'), ('bronze', '0.440'), ('vauxhall', '0.420')]
'king' -> [('saxon', '0.639'), ('crowned', '0.598'), ('edward', '0.528'), ('absorbed', '0.528'), ('supervised', '0.459')]
'queen' -> [('clapton', '0.665'), ('stormed', '0.660'), ('boudica', '0.644'), ('eric', '0.633'), ('kinks', '0.579')]


In [28]:
# Test analogies
def analogy(word_a, word_b, word_c):
    if word_a not in text_vocab or word_b not in text_vocab or word_c not in text_vocab:
        return None
    
    vec = final_embeddings[text_vocab[word_b]] - final_embeddings[text_vocab[word_a]] + final_embeddings[text_vocab[word_c]]
    
    similarities = []
    for w, idx in text_vocab.items():
        if w not in [word_a, word_b, word_c]:
            sim = cosine_similarity(vec, final_embeddings[idx])
            similarities.append((w, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[0]

print("=== Analogies ===")
result = analogy('king', 'man', 'queen')
if result:
    print(f"'king' - 'man' + 'queen' = '{result[0]}' ({result[1]:.3f})")

result = analogy('london', 'england', 'paris')
if result:
    print(f"'london' - 'england' + 'paris' = '{result[0]}' ({result[1]:.3f})")

=== Analogies ===
'london' - 'england' + 'paris' = 'disneyland' (0.854)


In [29]:
# Visualize in 3D with PCA
from sklearn.decomposition import PCA

top_words = [w for w, c in word_counts.most_common(200) if w in text_vocab]
top_indices = [text_vocab[w] for w in top_words]
top_embeddings = final_embeddings[top_indices]

pca = PCA(n_components=3)
embeddings_3d = pca.fit_transform(top_embeddings)

df = pd.DataFrame(embeddings_3d, columns=['x', 'y', 'z'])
df['word'] = top_words

fig = px.scatter_3d(df, x='x', y='y', z='z', text='word',
                 title='Word2Vec with Negative Sampling - Embeddings (PCA 3D)')
fig.update_traces(marker=dict(size=6), textposition='top center')
fig.update_layout(height=600, width=800)
fig.show()

In [30]:
# Summary
print("=== Summary ===")
print(f"Vocab size: {text_vocab_size}")
print(f"Embedding dim: {text_embed_dim}")
print(f"Positive pairs: {len(train_words)}")
print(f"Negative samples: {len(neg_labels)}")
print(f"Total training: {len(all_words)}")
print(f"Negative sampling ratio: {NEGATIVE_SAMPLING}:1")
print("\nModel uses:")
print("  - Two embeddings: target and context")
print("  - Dot product + sigmoid (binary classification)")
print("  - Much faster than full softmax!")

=== Summary ===
Vocab size: 3291
Embedding dim: 64
Positive pairs: 112368
Negative samples: 561840
Total training: 674208
Negative sampling ratio: 5:1

Model uses:
  - Two embeddings: target and context
  - Dot product + sigmoid (binary classification)
  - Much faster than full softmax!


In [31]:
# Guardar embeddings entrenados para uso futuro
np.save('target_embeddings.npy', model.target_embedding.get_weights()[0])
np.save('context_embeddings.npy', model.context_embedding.get_weights()[0])
np.save('text_vocab.npy', text_vocab)
print('Embeddings guardados como target_embeddings.npy y context_embeddings.npy')
print('Vocabulario guardado como text_vocab.npy')

Embeddings guardados como target_embeddings.npy y context_embeddings.npy
Vocabulario guardado como text_vocab.npy
